# Response Extraction

## Functions

In [ ]:
import json
from openai import AsyncOpenAI
import polars as pl
from tqdm.notebook import tqdm

from openai_wrapper.utils import jsonl2dict, save_cleaned
from utils.formatters import load_jsonl
from utils.gcs import list_from_gcs, load_from_gcs

In [5]:
from keys import OPENAI_KEY
client = AsyncOpenAI(api_key=OPENAI_KEY)

## Extract Data

In [6]:
case = 'us-tariffs'

In [10]:
# Transcripts
paths = [f'data/processed/{case}_transcripts_0_processed.jsonl']

transcripts = await jsonl2dict(load_jsonl(paths), client)
transcripts = save_cleaned(transcripts, f"data/extracted/{case}_transcripts_cleaned.json")

Processing Batches: 100%|██████████| 3/3 [00:00<00:00, 53.55batch/s]

0 errors found


In [21]:
# CommentThreads
paths = [f'data/processed/ {case}_commentThreads_{i}_processed.jsonl' for i in range(14)]
commentthreads = await jsonl2dict(load_jsonl(paths), client)
commentthreads = save_cleaned(commentthreads, f"data/extracted/{case}_commentThreads_cleaned.json")

Processing Batches: 100%|██████████| 338/338 [00:22<00:00, 14.71batch/s]
/var/folders/pr/x4pd5c4d2y94knncdskg53qm0000gn/T/ipykernel_766/1466942204.py:10: RuntimeWarning: coroutine 'fixErrorsOpenAI' was never awaited
  i['content'] = "Incorrect JSON format"


17 errors found


In [22]:
# CommentThreadsreplies
paths = [f'data/processed/ {case}_commentThreadsreplies_{i}_processed.jsonl' for i in range(4)]
commentthreadsreplies = await jsonl2dict(load_jsonl(paths), client)
commentthreadsreplies = save_cleaned(commentthreadsreplies, f"data/extracted/{case}_commentThreadsreplies_cleaned.json")

Processing Batches: 100%|██████████| 96/96 [00:01<00:00, 69.54batch/s]
/var/folders/pr/x4pd5c4d2y94knncdskg53qm0000gn/T/ipykernel_766/1466942204.py:10: RuntimeWarning: coroutine 'fixErrorsOpenAI' was never awaited
  i['content'] = "Incorrect JSON format"


2 errors found


In [ ]:
# Total Counts
len(transcripts), len(commentthreads), len(commentthreadsreplies)

(5874, 675486, 191187)

## Create consolidated Videos and Search Datasets

In [ ]:
with open('data/us-tariffs_transcripts.jsonl', 'r', encoding='utf-8') as file:
    filter_ids = [i['custom_id'] for i in [json.loads(line) for line in file if line.strip()]]

In [ ]:
def reshape_download(
    query: str,
    filename: str,
    bucket_name: str = 'youtube-us-tariffs2'
) -> dict:
    # Get list of files
    blobnames = list_from_gcs(bucket_name, '-'.join(query.split()).lower(), filename)

    # Relevant columns
    if filename == 'search.csv':
        cols = [
            'videoId', 'channelId', 'channelTitle', 'title', 'description',
            'publishTime', 'publishedAt', 'liveBroadcastContent'
        ]
    elif filename == 'videos.csv':
        cols = [
            'id', 'viewCount', 'favoriteCount', 'likeCount', 'commentCount', 'topicCategories'
        ]
    elif filename == 'commentThreads.csv':
        cols = [
            'id', 'channelId', 'videoId', 'authorDisplayName', 
            'textDisplay', 'totalReplyCount', 'likeCount', 
            'updatedAt', 'publishedAt'
        ]
    elif filename == 'commentThreadsreplies.csv':
        cols = [
            'id', 'parentId', 'channelId', 'videoId', 'authorDisplayName',
            'textDisplay', 'likeCount', 'updatedAt', 'publishedAt'
        ]
    elif filename == 'transcripts.csv':
        cols = [
            'videoId', 'language', 'is_generated', 'transcript'
        ]
    else:
        raise ValueError(f"Unknown filename type: {filename}")

    id_col = 'id' if filename == 'videos.csv' else 'videoId'

    # Download from GCS
    ids = set()
    all_data = []

    for blob in tqdm(blobnames, desc="Downloading blobs", unit="blob", total=len(blobnames)):
        data = load_from_gcs(bucket_name, blob_name=blob)
        if data:
            all_data += [x for x in data if x.get(id_col) in filter_ids]

    # Save to Parquet using Polars
    if all_data:
        df = pl.DataFrame(all_data)
        df = df.select([col for col in df.columns if col in cols])  # Keep only relevant cols

        # Save as Parquet
        out_path = f"data/{filename.replace('.csv', '')}.parquet"
        df.write_parquet(out_path)
        print(f"Saved to {out_path}")

In [ ]:
query = 'US Tariffs'

reshape_download(query, 'search.csv')
reshape_download(query, 'videos.csv')
reshape_download(query, 'commentThreads.csv')
reshape_download(query, 'commentThreadsreplies.csv')
reshape_download(query, 'transcripts.csv')

## Join Data

### Cleaning up

In [5]:
import json

with open("data/extracted/us-tariffs_transcripts_cleaned.json", "r", encoding="utf-8") as f:
    transcripts = json.load(f)

with open("data/extracted/us-tariffs_commentThreads_cleaned.json", "r", encoding="utf-8") as f:
    commentthreads = json.load(f)

with open("data/extracted/us-tariffs_commentThreadsreplies_cleaned.json", "r", encoding="utf-8") as f:
    commentthreadsreplies = json.load(f)

In [ ]:
# Reformat data to fit in table format
def reformat_data(data: list) -> list:
    output = []
    incorrect = 0
    for item in data:
        if item['content'] != "Incorrect JSON format":
            for emotopic in item['content']['emotopic']:
                if emotopic:
                    output.append(
                        {
                            "id": item['id'],
                            "entity": emotopic['entity'],
                            'topic': emotopic['topic'],
                            'emotion': emotopic['emotion'],
                            'topic_explanation': emotopic['topic_explanation'],
                            'emotion_explanation': emotopic['emotion_explanation'],
                        }
                    )
        else:
            incorrect += 1
    print(f"Incorrect JSON format: {incorrect} out of {len(data)}")
    return output

transcripts_reformatted = reformat_data(transcripts)
commentthreads_reformatted = reformat_data(commentthreads)
commentthreadsreplies_reformatted = reformat_data(commentthreadsreplies)

Incorrect JSON format: 0 out of 5874
Incorrect JSON format: 17 out of 675486
Incorrect JSON format: 2 out of 191187


In [ ]:
# Filter columns and fix data types
df_search = pl.read_parquet(f"data/youtube/search.parquet")
df_videos = pl.read_parquet(f"data/youtube/videos.parquet")
df_commentthreads = pl.read_parquet(f"data/youtube/commentThreads.parquet")
df_commentthreadsreplies = pl.read_parquet(f"data/youtube/commentThreadsreplies.parquet")
df_transcripts = pl.read_parquet(f"data/youtube/transcripts.parquet")

with open("data/youtube/durations.json") as f:
    raw = json.load(f)
df_durations = pl.DataFrame([{"videoId": k, "duration": v} for k, v in raw.items()])

df_search = df_search\
    .select([
        pl.col("videoId"),
        pl.col("channelId").alias("channelID"),
        pl.col("channelTitle"),
        pl.col("title"),
        pl.col("description"),
        pl.col("publishedAt").cast(pl.Datetime),
    ])\
    .unique(subset=["videoId"])

df_videos = df_videos\
    .select([
        pl.col("id").alias("videoId"),
        pl.col("viewCount"),
        pl.col("likeCount"),
        pl.col("commentCount"),
        pl.col("topicCategories"),
    ]).\
    with_columns(
        pl.col("topicCategories").str.split("|")
    )\
    .unique(subset=["videoId"])

df_commentthreads = df_commentthreads\
    .select([
        pl.col("id").alias("commentThreadId"),
        pl.col("videoId"),
        pl.col("channelId"),
        pl.col("authorDisplayName"),
        pl.col("publishedAt").cast(pl.Datetime),
        pl.col("likeCount"),
        pl.col("totalReplyCount"),
        pl.col("textDisplay").alias("text")
    ])\
    .unique(subset=["commentThreadId"])

df_commentthreadsreplies = df_commentthreadsreplies\
    .select([
        pl.col("id").alias("commentThreadReplyId"),
        pl.col("parentId").alias("commentThreadId"),
        pl.col("authorDisplayName"),
        pl.col("publishedAt").cast(pl.Datetime),
        pl.col("likeCount"),
        pl.col("textDisplay").alias("text")
    ])\
    .unique(subset=["commentThreadReplyId"])

In [7]:
# Rename columns for 
df_transcripts_processed = pl.DataFrame(transcripts_reformatted)
df_commentthreads_processed = pl.DataFrame(commentthreads_reformatted)
df_commentthreadsreplies_processed = pl.DataFrame(commentthreadsreplies_reformatted)

df_transcripts_processed = df_transcripts_processed.rename({
    "id": "videoId"
})
df_commentthreads_processed = df_commentthreads_processed.rename({
    "id": "commentThreadId"
})
df_commentthreadsreplies_processed = df_commentthreadsreplies_processed.rename({
    "id": "commentThreadReplyId"
})

### Search-Videos-Transcripts

In [8]:
# Join dataframes
df_videos_full = df_transcripts_processed.join(
    df_search,
    on="videoId",
    how="left"
).join(
    df_videos,
    on="videoId",
    how="left"
).join(
    df_durations,
    on="videoId",
    how="left"
)

In [14]:
df_commentthreads_full = df_commentthreads_processed.join(
    df_commentthreads.rename({
        col: col + '_ct' for col in df_commentthreads.columns if col not in ['commentThreadId', 'videoId']
    }),
    on="commentThreadId",
    how="left"
).join(
    df_videos_full.rename({
        col: col + '_videos' for col in df_videos_full.columns if col != 'videoId'
    }),
    on="videoId",
    how="left"
)

In [39]:
df_commentthreads_replies_full = df_commentthreadsreplies_processed.join(
    df_commentthreadsreplies.rename({
        col: col + '_ctr' for col in df_commentthreadsreplies.columns if col not in ['commentThreadReplyId', 'commentThreadId']
    }),
    on="commentThreadReplyId",
    how="left"
).join(
    df_commentthreads_full.rename({
        col: col + '_ctf' for col in df_commentthreads_full.columns if col not in ['commentThreadId', 'videoId']
    }),
    on="commentThreadId",
    how="left"
)

In [47]:
missing_videos = len(df_videos_full.filter(pl.col("videoId").is_null()))
print(f"Missing videos: {missing_videos}")
missing_commentthreads = len(df_commentthreads_full.filter(pl.col("videoId").is_null()))
print(f"Missing comment threads: {missing_commentthreads}")
missing_commentthreads_replies = len(df_commentthreads_replies_full.filter(pl.col("videoId").is_not_null(), pl.col("commentThreadId").is_null()))
print(f"Missing comment thread replies: {missing_commentthreads_replies}")

# Save the final dataframes
df_videos_full.write_parquet("data/parquet/videos_full.parquet")
df_commentthreads_full.write_parquet("data/parquet/commentThreads_full.parquet")
df_commentthreads_replies_full.filter(pl.col("videoId").is_not_null()).write_parquet("data/parquet/commentThreadsreplies_full.parquet")

Missing videos: 0
Missing comment threads: 0
Missing comment thread replies: 0
